In [2]:
!pip install fastparquet

You should consider upgrading via the 'c:\users\khann\appdata\local\programs\python\python38\python.exe -m pip install --upgrade pip' command.


In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Generate sample data
df = pd.read_parquet("../../../data/sales_features.parquet").sort_values(['store', 'date'])


In [6]:
for col in df.columns:
    print(f"Column: {col}, Type: {df[col].dtype}")

Column: date, Type: datetime64[ns]
Column: name, Type: object
Column: address, Type: object
Column: city, Type: object
Column: zipcode, Type: float64
Column: county, Type: object
Column: item_details, Type: object
Column: lon, Type: float64
Column: lat, Type: float64
Column: sale_bottles, Type: int64
Column: sale_bottles_mean, Type: float64
Column: sale_bottles_median, Type: float64
Column: sale_bottles_std, Type: float64
Column: sale_bottles_min, Type: int64
Column: sale_bottles_max, Type: int64
Column: sale_bottles_skew, Type: float64
Column: sale_bottles_sem, Type: float64
Column: sale_dollars, Type: float64
Column: sale_dollars_mean, Type: float64
Column: sale_dollars_median, Type: float64
Column: sale_dollars_std, Type: float64
Column: sale_dollars_min, Type: float64
Column: sale_dollars_max, Type: float64
Column: sale_dollars_skew, Type: float64
Column: sale_dollars_sem, Type: float64
Column: sale_liters, Type: float64
Column: sale_liters_mean, Type: float64
Column: sale_liters_m

In [4]:
from typing import Dict, List


def add_known_future(
    df: pd.DataFrame, 
    new_col: str,
    shift: int,
    compute_fn: callable,
    group_col: str = 'store'
) -> pd.DataFrame:
    """
    Creates future-known features using row-wise computations
    
    Args:
        df: Input DataFrame
        new_col: Name for new future feature column
        shift: Number of periods to shift (-1 for next day)
        compute_fn: Lambda function that takes row and returns future value
        group_col: Column to group by for temporal shifts
    
    Returns:
        DataFrame with new future feature column
    """
    df = df.copy()
    # Compute future value using current row's information
    df[new_col] = df.apply(compute_fn, axis=1)
    # Shift within groups to align with target
    df[new_col] = df.groupby(group_col)[new_col].shift(shift)
    return df

def create_lag_features(
    df: pd.DataFrame,
    features: Dict[str, List[str]],
    look_back: int,
    group_col: str = 'store'
) -> pd.DataFrame:
    """
    Creates lag features for specified observed features
    
    Args:
        df: Input DataFrame
        features: Dictionary containing feature categories
        look_back: Number of lookback periods to create
        group_col: Column to group by for temporal shifts
    
    Returns:
        DataFrame with added lag features
    """
    df = df.copy()
    for feat in features['observed']:
        for lag in range(look_back):
            new_col = f'{feat}_lag_{lag}'
            df[new_col] = df.groupby(group_col)[feat].shift(lag)
            features['observed'].append(new_col)
    return df.dropna()

In [26]:
features = {
    'static': [
        'store', 'name', 'address', 'city', 'zipcode', 'county', 
        'lon', 'lat', 'store_size', 'store_avg_sales', 'city_avg_sales',
        'county_avg_sales', 'store_avg_transactions', 'store_to_city_sales_ratio',
        'store_to_county_sales_ratio', 'store_avg_items'
    ],
    'observed': [
        # Core metrics
        'sale_dollars', 'sale_bottles', 'sale_liters', 'sale_gallons',
        'transaction_count', 'unique_transactions',
        
        # Statistical features
        'sale_dollars_mean', 'sale_dollars_median', 'sale_dollars_std',
        'sale_dollars_skew', 'sale_dollars_sem',
        
        # Temporal encodings
        'day_of_week', 'month', 'quarter', 'year', 'week_of_year',
        'month_sin', 'month_cos', 'day_of_month_sin', 'day_of_month_cos',
        
        # Product metrics
        'unique_categories', 'category_count', 'unique_items',
        
        # Price metrics
        'avg_price_per_bottle', 'avg_price_per_liter', 'profit_margin',
        'discount_factor', 'avg_transaction_value',
        
        # Rolling features (keep these as they contain aggregated history)
        'sale_dollars_rolling_mean_7D', 'sale_dollars_rolling_std_7D',
        'sale_dollars_rolling_max_7D', 'sale_dollars_rolling_min_7D'
    ],
    'known_future': [
        # Will be populated using the function below
    ]
}


def add_future_features(
    df: pd.DataFrame,
    date_col: str = 'date',
    group_col: str = 'store',
    days_ahead: int = 1
) -> pd.DataFrame:
    
    df = df.copy()
    
    # Create future date reference
    df['future_date'] = df[date_col] + pd.Timedelta(days=days_ahead)
    
    # Calendar features
    df['is_weekend_future'] = df['future_date'].dt.weekday >= 5
    # df['is_holiday_future'] = df['future_date'].isin(holiday_dates)  # Replace with your holiday list
    df['month_future'] = df['future_date'].dt.month
    df['day_of_week_future'] = df['future_date'].dt.weekday
    
    # Special day flags
    df['is_end_of_month_future'] = df['future_date'].dt.is_month_end
    df['is_quarter_end_future'] = df['future_date'].dt.is_quarter_end
    
    # Cyclical encoding for future date
    df['month_sin_future'] = np.sin(2 * np.pi * df['month_future']/12)
    df['month_cos_future'] = np.cos(2 * np.pi * df['month_future']/12)
    
    # Drop temporary column
    df.drop('future_date', axis=1, inplace=True)
    
    # Add to known future features
    features['known_future'].extend([
        'is_weekend_future', 
        # 'is_holiday_future', 
        'month_future',
        'day_of_week_future', 'is_end_of_month_future', 'is_quarter_end_future',
        'month_sin_future', 'month_cos_future'
    ])
    
    return df



In [27]:
# Before model training
df = add_future_features(df)

# Verify features
print("Known future features:", features['known_future'])

Known future features: ['is_weekend_future', 'month_future', 'day_of_week_future', 'is_end_of_month_future', 'is_quarter_end_future', 'month_sin_future', 'month_cos_future']


In [28]:
from typing import Dict, List

def generate_dynamic_lags(
    df: pd.DataFrame,
    base_features: List[str],
    lookback_config: Dict[str, List[int]],
    group_col: str = 'store'
) -> pd.DataFrame:
    """
    Creates lag features for multiple base features with different lookbacks
    
    Args:
        df: Input DataFrame
        base_features: List of features to create lags for
        lookback_config: Dictionary of {lookback_type: list_of_days}
            Example: {'lag': [1, 7], 'rolling_mean': [7, 14]}
        group_col: Grouping column for temporal operations
            
    Returns:
        DataFrame with added lag features
    """
    df = df.copy()
    
    for feature in base_features:
        for operation, windows in lookback_config.items():
            for window in windows:
                if operation == 'lag':
                    df[f'{feature}_lag_{window}d'] = df.groupby(group_col)[feature].shift(window)
                    features['observed'].append(f'{feature}_lag_{window}d')
                
                elif operation == 'rolling_mean':
                    df[f'{feature}_rollmean_{window}d'] = (
                        df.groupby(group_col)[feature]
                        .transform(lambda x: x.rolling(window, min_periods=1).mean())
                    )
                    features['observed'].append(f'{feature}_rollmean_{window}d')
    
    return df.dropna()

# Example usage
df = generate_dynamic_lags(
    df,
    base_features=['sale_dollars', 'transaction_count'],
    lookback_config={
        'lag': [1, 3, 7],
        'rolling_mean': [7, 14]
    }
)

In [29]:
df.tail(5)

,date,name,address,city,zipcode,county,item_details,lon,lat,sale_bottles,...,month_sin_future,month_cos_future,sale_dollars_lag_3d,sale_dollars_rollmean_7d,sale_dollars_rollmean_14d,transaction_count_lag_1d,transaction_count_lag_3d,transaction_count_lag_7d,transaction_count_rollmean_7d,transaction_count_rollmean_14d
store,,,,,,,,,,,,,,,,,,,,,
10443,2025-01-02,SIP & BURN / ANKENY,1200 NW 36TH ST STE 101,ANKENY,50023.0,POLK,[{'category_name': 'AMERICAN CORDIALS & LIQUEU...,-93.6124,41.76082,102,...,0.5,0.866025,143.79,954.332857,778.058571,1.0,2.0,7.0,9.571429,8.571429
10443,2025-01-10,SIP & BURN / ANKENY,1200 NW 36TH ST STE 101,ANKENY,50023.0,POLK,[{'category_name': 'AMERICAN CORDIALS & LIQUEU...,-93.6124,41.76082,5,...,0.5,0.866025,1678.60,944.961429,752.060000,21.0,16.0,1.0,9.571429,8.214286
10443,2025-01-15,SIP & BURN / ANKENY,1200 NW 36TH ST STE 101,ANKENY,50023.0,POLK,[{'category_name': 'TEMPORARY & SPECIALTY PACK...,-93.6124,41.76082,1,...,0.5,0.866025,78.72,802.410000,688.277143,1.0,1.0,12.0,8.000000,7.428571
10443,2025-01-22,SIP & BURN / ANKENY,1200 NW 36TH ST STE 101,ANKENY,50023.0,POLK,"[{'category_name': 'IMPORTED SCHNAPPS', 'im_de...",-93.6124,41.76082,12,...,0.5,0.866025,2323.80,674.544286,612.820714,1.0,21.0,14.0,6.428571,6.714286
10443,2025-01-29,SIP & BURN / ANKENY,1200 NW 36TH ST STE 101,ANKENY,50023.0,POLK,"[{'category_name': 'AMERICAN SCHNAPPS', 'im_de...",-93.6124,41.76082,30,...,0.5,0.866025,46.90,721.181429,621.552857,3.0,1.0,2.0,7.428571,7.142857


In [31]:

# Train-test split
train_df = df.groupby('store', group_keys=False).apply(lambda x: x.iloc[:-7]).reset_index()
test_df = df.groupby('store', group_keys=False).apply(lambda x: x.iloc[-7:]).reset_index()

print(train_df.columns)

# Preprocessing.
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_df[features['observed']])
test_scaled = scaler.transform(test_df[features['observed']])

# Create sequences
def create_sequences(data, look_back=7):
    sequences = []
    for i in range(len(data) - look_back):
        sequences.append(data[i:i+look_back])
    return np.array(sequences)

X_train_seq = create_sequences(train_scaled)
X_test_seq = create_sequences(test_scaled)

# Prepare inputs
def prepare_inputs(dff, seq):
    static = pd.Categorical(dff['store']).codes[seq.shape[1]:]  # Now 'store' is a column
    known_future = dff[features['known_future']].values[seq.shape[1]:]
    return (static[:len(seq)], known_future[:len(seq)]), dff['sale_dollars'].values[seq.shape[1]:]

(train_static, train_known), y_train = prepare_inputs(train_df, X_train_seq)
(test_static, test_known), y_test = prepare_inputs(test_df, X_test_seq)


Index(['store', 'date', 'name', 'address', 'city', 'zipcode', 'county',
       'item_details', 'lon', 'lat',
       ...
       'month_sin_future', 'month_cos_future', 'sale_dollars_lag_3d',
       'sale_dollars_rollmean_7d', 'sale_dollars_rollmean_14d',
       'transaction_count_lag_1d', 'transaction_count_lag_3d',
       'transaction_count_lag_7d', 'transaction_count_rollmean_7d',
       'transaction_count_rollmean_14d'],
      dtype='object', length=167)


In [ ]:

# TFT Model Components
def tft_model(look_back=7, num_static=3, num_observed=3, num_known=2):
    # Inputs
    static_input = Input(shape=(1,), name='static_input')
    observed_input = Input(shape=(look_back, num_observed), name='observed_input')
    known_input = Input(shape=(num_known,), name='known_input')
    
    # Static embeddings
    static_embedding = Embedding(num_static, 4)(static_input)
    static_vars = Reshape((4,))(static_embedding)
    
    # Temporal processing
    lstm_layer = LSTM(32, return_sequences=True)(observed_input)
    attn_layer = MultiHeadAttention(num_heads=2, key_dim=8)(lstm_layer, lstm_layer)
    temporal_features = LayerNormalization()(lstm_layer + attn_layer)
    temporal_features = LSTM(16)(temporal_features)
    
    # Feature fusion
    combined = Concatenate()([static_vars, temporal_features, known_input])
    combined = Dense(32, activation='relu')(combined)
    combined = Dropout(0.1)(combined)
    
    # Output
    output = Dense(1)(combined)
    
    return Model(inputs=[static_input, observed_input, known_input], outputs=output)

# Initialize and compile model
model = tft_model(
    look_back=7,
    num_static=len(stores),
    num_observed=len(features['observed']),
    num_known=len(features['known_future'])
)
model.compile(optimizer='adam', loss='mse')

# Train the model
history = model.fit(
    [train_static, X_train_seq, train_known],
    y_train,
    validation_data=([test_static, X_test_seq, test_known], y_test),
    epochs=20,
    batch_size=32,
    verbose=1
)

# Example prediction
sample_idx = 0
prediction = model.predict([
    np.array([test_static[sample_idx]]),
    X_test_seq[sample_idx][np.newaxis, ...],
    np.array([test_known[sample_idx]])
])
print(f"Predicted sales: {prediction[0][0]:.2f}, Actual sales: {y_test[sample_idx]:.2f}")